In [2]:
!pip uninstall -y torchvision torchaudio && \
pip install -U --upgrade-strategy eager \
    langchain \
    "langchain-openrouter" \
    langchain-community \
    langchain-text-splitters \
    langchain-chroma \
    langchain-huggingface \
    deepagents \
    sentence-transformers \
    transformers

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
  Using cached websockets-17.0.1-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.3 kB)
  Using cached websockets-16.1.1-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.0/738.0 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.10.18
    Uninstalling langsmith-0.10.18:
      Successfully uninstalled langsmith-0.10.18


In [3]:
import os

os.environ["OPENROUTER_API_KEY"] = "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"
os.environ["PINECONE_API_KEY"] = "pcsk_625YqQ_BYd1edGh8Yq3Mp3cPKg5ZGxrSVXy8F13ZSEUZoZdVyuDB6DXp27LaLqAKU274Eh"
os.environ["INDEX_NAME"] = "medium-blogs-embeddings-index"

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("/content/textfile.txt", encoding="utf-8")

documents = loader.load()

print(documents)
print(documents[0].page_content)
print(documents[0].metadata)

/tmp/ipykernel_10934/3887791823.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': '/content/textfile.txt'}, page_content='Vector Database: What is it and why you should know it?\nEjiro Onose\nEjiro Onose\n\nFollow\n9 min read\n·\nDec 22, 2023\n347\n\n\n1\n\n\n\nPress enter or click to view image in full size\n\n“If 2021 was the year of graph databases, 2023 is the year of vector databases” — Chip Huen.\n\nWhy? you ask...\n\nWell, it’s because Generative AI and Large Language Models (LLMs) are now popular and one of the best ways to handle LLM data is with a vector database because vector databases provide the ideal infrastructure for managing the complex, high-dimensional data that LLMs produce and thrive on.\n\nIn this article, we’ll discuss what vector databases are, how they work, and some excellent vector database tools you should check out.\n\nBefore we dive into vector databases, let’s first understand what a vector is.\n\nWhat is a Vector?\nIn machine learning (ML), a vector is a collection of numerical values that represent the 

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

Documents: 1
Chunks: 19


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [7]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [8]:
vector_store.add_documents(documents=chunks)
print(f"Indexed {len(chunks)} chunks.")

Indexed 19 chunks.


In [9]:
len(chunks)

19

In [10]:
type(chunks[0])

langchain_core.documents.base.Document

In [11]:
from langchain.tools import tool


@tool
def search_documentation(query: str) -> str:
    """Search the documentation for relevant information."""
    docs = vector_store.similarity_search(query, k=4)

    return "\n\n---\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n"
        f"{doc.page_content}"
        for doc in docs
    )

In [12]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(model='auto')

In [13]:

from deepagents import create_deep_agent


# ---------------------------------------------------------
# 2. System prompt
# ---------------------------------------------------------

SYSTEM_PROMPT = """
You are a documentation-based RAG assistant.

Your job is to answer the user's questions using the available
documentation.

## Rules

1. Always use the `search_documentation` tool when the user's
   question requires information from the documentation.

2. Do not rely on your own knowledge when answering questions
   about the documentation.

3. Carefully read the retrieved documents and use them as the
   primary source for your answer.

4. If the retrieved documentation does not contain enough
   information to answer the question, clearly say that the
   documentation does not provide enough information.

5. Never invent APIs, configuration options, functions, behavior,
   or facts that are not supported by the retrieved documentation.

6. If multiple retrieved documents are relevant, synthesize
   the information from them.

7. Keep answers clear and concise.

8. When possible, mention the source of the information.

9. If the user asks a general conversational question that does
   not require documentation, you may answer normally without
   using the retrieval tool.
"""


# ---------------------------------------------------------
# 3. Create Deep Agent
# ---------------------------------------------------------

agent = create_deep_agent(
    model=llm,
    tools=[search_documentation],
    system_prompt=SYSTEM_PROMPT,
)

In [14]:
from langchain_core.messages import HumanMessage

messages = []

while True:
    user_input = input("You: ")

    if user_input.lower() in {"exit", "quit"}:
        break

    messages.append(HumanMessage(content=user_input))

    result = agent.invoke({
        "messages": messages
    })

    messages = result["messages"]

    # The last message should be the final AI response
    print(f"AI: {messages[-1].content}")

You: What is pinecone ? 
AI: Based on the documentation, Pinecone is described as a fast and scalable platform for efficient retrieval of similar data points based on their vector representations. Key characteristics from the documentation:

- **Scale**: Handles large-scale ML applications with millions or billions of data points.
- **Infrastructure management**: Provides infrastructure management and maintenance to its users.
- **Performance**: Handles high query throughput with low-latency search.
- **Security**: A secure platform that meets the security needs of businesses and organizations.
- **Ease of use**: User-friendly, with a simple API for storing and retrieving vector data, making it easy to integrate into existing ML workflows.
- **Real-time updates**: Supports real-time updates so the vector database remains up-to-date and accurate as new data points are added.
- **Integrations**: Can be synced with data from various sources using tools like Airbyte and monitored using Dat